In [2]:
!git clone https://github.com/abolfazlaghdaee/Irony_Detection

Cloning into 'Irony_Detection'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 129 (delta 21), reused 25 (delta 6), pack-reused 77 (from 1)
Receiving objects: 100% (129/129), 83.59 MiB | 14.50 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [3]:
%cd Irony_Detection

/content/Irony_Detection


In [62]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split



In [29]:
df = pd.read_csv("data/Preprocessed_data.csv")
df.head()

,tweet_with_emoji_meaning,label
0,پیرمرد وصیت احدی تاکید احدی درد دل نکنید بعدش ...,0
1,مجوز بده ملت ماشین استاندارد بتونن بیارن سوار...,0
2,دیت دختره دید زشتم می‌خواست پاشه بره آینهی دست...,1
3,اکیپ دخترونه هست پایه قراراست کلاس نمیذاره زود...,0
4,۵۰۰ نفری مراسم سالگرد پدر همسرم شرکت فوتی‌ی ۴۰...,0


In [30]:
dataset = Dataset.from_pandas(df)

In [31]:
model_name = "HooshvareLab/bert-base-parsbert-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

In [32]:
def bert_tokenize(batch):
    return tokenizer(batch['tweet_with_emoji_meaning'], padding="max_length", truncation=True, max_length=128)


bert_tokenized_dataset = dataset.map(bert_tokenize, batched=True)

Map:   0%|          | 0/14676 [00:00<?, ? examples/s]

In [33]:
bert_tokenized_dataset = bert_tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = bert_tokenized_dataset["train"]
test_dataset = bert_tokenized_dataset["test"]

In [34]:
num_labels = len(set(df['label']))
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [37]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=bert_tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-38-1864845588.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abolfzl (abolfzl-university-of-kashan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.469200,0.464695,0.802793,0.802868,0.802793,0.802674
2,0.357300,0.658657,0.773842,0.789695,0.773842,0.769574


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.469200,0.464695,0.802793,0.802868,0.802793,0.802674
2,0.357300,0.658657,0.773842,0.789695,0.773842,0.769574
3,0.255700,0.927314,0.805518,0.805669,0.805518,0.805368
4,0.137400,1.111480,0.799728,0.799796,0.799728,0.799748


TrainOutput(global_step=5872, training_loss=0.3035386972921096, metrics={'train_runtime': 1848.9286, 'train_samples_per_second': 25.398, 'train_steps_per_second': 3.176, 'total_flos': 3088923789926400.0, 'train_loss': 0.3035386972921096, 'epoch': 4.0})

### XLM Roberta

In [71]:
tokenizer = AutoTokenizer.from_pretrained("classla/xlm-roberta-base-multilingual-text-genre-classifier")

In [72]:
def xlm_tokenizer(batch):
  return tokenizer(batch['tweet_with_emoji_meaning'], padding = 'max_length', truncation = True, max_length =128)

In [73]:
dataset  = pd.read_csv("data/Preprocessed_data.csv")
df.head()

,tweet_with_emoji_meaning,label
0,پیرمرد وصیت احدی تاکید احدی درد دل نکنید بعدش ...,0
1,مجوز بده ملت ماشین استاندارد بتونن بیارن سوار...,0
2,دیت دختره دید زشتم می‌خواست پاشه بره آینهی دست...,1
3,اکیپ دخترونه هست پایه قراراست کلاس نمیذاره زود...,0
4,۵۰۰ نفری مراسم سالگرد پدر همسرم شرکت فوتی‌ی ۴۰...,0


In [74]:
dataset = Dataset.from_pandas(df)

In [75]:
xlm_tokenized_dataset = dataset.map(xlm_tokenizer)

Map:   0%|          | 0/14676 [00:00<?, ? examples/s]

In [76]:
xlm_tokenized_dataset = xlm_tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = xlm_tokenized_dataset["train"]
test_dataset = xlm_tokenized_dataset["test"]
num_labels = len(set(df['label']))

In [77]:
num_labels

2

In [79]:
from transformers import XLMRobertaForSequenceClassification

xlm_model = AutoModelForSequenceClassification.from_pretrained("classla/xlm-roberta-base-multilingual-text-genre-classifier", num_labels=2)

RuntimeError: Error(s) in loading state_dict for Linear:
	size mismatch for bias: copying a param with shape torch.Size([9]) from checkpoint, the shape in current model is torch.Size([2]).

In [80]:
training_xlm_args = TrainingArguments(output_dir = './xlm_results',
                                 eval_strategy='epoch',
                                 save_strategy="epoch",
                                  num_train_epochs=4,
                                 learning_rate = 2e-5,
                                 weight_decay = 0.01,
                                per_device_train_batch_size = 8,
                                 per_device_eval_batch_size = 8,
                                  load_best_model_at_end=True,
    metric_for_best_model="accuracy"
                                      )

In [81]:
xlm_trainer= Trainer(model = xlm_model,
                   args = training_xlm_args,
                   train_dataset = train_dataset,
                   eval_dataset = test_dataset,
                   tokenizer = tokenizer,
                   compute_metrics = compute_metrics)

/tmp/ipython-input-81-2424609941.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  xlm_trainer= Trainer(model = xlm_model,


In [82]:
xlm_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.608200,0.656050,0.687670,0.733112,0.687670,0.679031
2,0.491000,0.502818,0.787466,0.796951,0.787466,0.787341
3,0.473400,0.505111,0.798025,0.797852,0.798025,0.797837


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.608200,0.656050,0.687670,0.733112,0.687670,0.679031
2,0.491000,0.502818,0.787466,0.796951,0.787466,0.787341
3,0.473400,0.505111,0.798025,0.797852,0.798025,0.797837
4,0.460400,0.492734,0.800409,0.802546,0.800409,0.800657


TrainOutput(global_step=5872, training_loss=0.4973260999050712, metrics={'train_runtime': 1873.8756, 'train_samples_per_second': 25.06, 'train_steps_per_second': 3.134, 'total_flos': 3088923789926400.0, 'train_loss': 0.4973260999050712, 'epoch': 4.0})